# Setup

In [15]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Baseline Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Proposed Ensemble Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [2]:
def print_evaluation_metrics(model_name, y_true, y_pred, y_prob):
    print(f"=== Performance: {model_name} ===")
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred):.4f}")
    print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score  : {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC   : {roc_auc_score(y_true, y_prob):.4f}\n")

In [3]:
current_dir = os.getcwd()
print(f"Started in: {current_dir}")

if current_dir.endswith('notebook'): 
    os.chdir('..')

print(f"Now running in ROOT: {os.getcwd()}")

Started in: c:\Users\samue\OneDrive\Documents\GitHub\TangerangCast\notebook
Now running in ROOT: c:\Users\samue\OneDrive\Documents\GitHub\TangerangCast


In [4]:
# read data for baseline models (distance based)
hist_processed_path = "data/processed/scaledhistoric.csv" 
df_hist_processed = pd.read_csv(hist_processed_path)
df_hist_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 236952 entries, 0 to 236951
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Temperature  236952 non-null  float64
 1   Humidity     236952 non-null  float64
 2   Cloud_Cover  236952 non-null  float64
 3   Pressure     236952 non-null  float64
 4   Wind_Speed   236952 non-null  float64
 5   Latitude     236952 non-null  float64
 6   Longitude    236952 non-null  float64
 7   Hour         236952 non-null  float64
 8   Month        236952 non-null  float64
 9   Rain         236952 non-null  int64  
dtypes: float64(9), int64(1)
memory usage: 18.1 MB


In [5]:
# read data for proposed ensemble models (decision tree based)
hist_path = "data/raw/historic.csv" 
df_hist = pd.read_csv(hist_path)
df_hist.info()

<class 'pandas.DataFrame'>
RangeIndex: 236952 entries, 0 to 236951
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Timestamp    236952 non-null  str    
 1   Temperature  236952 non-null  float64
 2   Humidity     236952 non-null  int64  
 3   Cloud_Cover  236952 non-null  int64  
 4   Pressure     236952 non-null  float64
 5   Wind_Speed   236952 non-null  float64
 6   Location     236952 non-null  str    
 7   Latitude     236952 non-null  float64
 8   Longitude    236952 non-null  float64
 9   Rain         236952 non-null  int64  
dtypes: float64(5), int64(3), str(2)
memory usage: 23.8 MB


In [8]:
# dropping unused features (including the target feature 'Rain')
processed_columns_to_drop = ['Rain', 'Hour', 'Month', 'Latitude', 'Longitude', 'Timestamp', 'Location']
x_processed = df_hist.drop(columns=processed_columns_to_drop, errors='ignore')
x_processed.info()
y_processed = df_hist['Rain']

# Split into training and internal validation sets
X_processed_train, X_processed_val, y_processed_train, y_processed_val = train_test_split(x_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed)
X_processed_train = X_processed_train.sample(n=20000, random_state=42)
y_processed_train = y_processed_train.loc[X_processed_train.index]

<class 'pandas.DataFrame'>
RangeIndex: 236952 entries, 0 to 236951
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Temperature  236952 non-null  float64
 1   Humidity     236952 non-null  int64  
 2   Cloud_Cover  236952 non-null  int64  
 3   Pressure     236952 non-null  float64
 4   Wind_Speed   236952 non-null  float64
dtypes: float64(3), int64(2)
memory usage: 9.0 MB


In [9]:
columns_to_drop = ['Rain', 'Timestamp', 'Location', 'Latitude', 'Longitude']
X = df_hist.drop(columns=columns_to_drop, errors='ignore')
X.info()
y = df_hist['Rain']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = X_train.sample(n=20000, random_state=42)
y_train = y_train.loc[X_train.index]

<class 'pandas.DataFrame'>
RangeIndex: 236952 entries, 0 to 236951
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Temperature  236952 non-null  float64
 1   Humidity     236952 non-null  int64  
 2   Cloud_Cover  236952 non-null  int64  
 3   Pressure     236952 non-null  float64
 4   Wind_Speed   236952 non-null  float64
dtypes: float64(3), int64(2)
memory usage: 9.0 MB


# Train Baseline Models

In [10]:
print("--- TRAINING LOGISTIC REGRESSION ---")
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_processed_train, y_processed_train)
logistic_pred = logistic_model.predict(X_processed_val)
logistic_prob = logistic_model.predict_proba(X_processed_val)[:, 1]
print_evaluation_metrics("Logistic Regression", y_processed_val, logistic_pred, logistic_prob)

--- TRAINING LOGISTIC REGRESSION ---
=== Performance: Logistic Regression ===
Accuracy  : 0.6947
Precision : 0.5517
Recall    : 0.2648
F1-Score  : 0.3578
ROC-AUC   : 0.7158



In [11]:
print("--- TRAINING SUPPORT VECTOR MACHINE ---")
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_processed_train, y_processed_train)
svm_pred = svm_model.predict(X_processed_val)
svm_prob = svm_model.predict_proba(X_processed_val)[:, 1]
print_evaluation_metrics("Support Vector Machine", y_processed_val, svm_pred, svm_prob)

--- TRAINING SUPPORT VECTOR MACHINE ---
=== Performance: Support Vector Machine ===
Accuracy  : 0.6788
Precision : 0.0000
Recall    : 0.0000
F1-Score  : 0.0000
ROC-AUC   : 0.6423



c:\Users\samue\OneDrive\Documents\GitHub\TangerangCast\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [12]:
print("--- TRAINING RANDOM FOREST ---")
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_processed_train, y_processed_train)
rf_pred = rf_model.predict(X_processed_val)
rf_prob = rf_model.predict_proba(X_processed_val)[:, 1]
print_evaluation_metrics("Random Forest", y_processed_val, rf_pred, rf_prob)

--- TRAINING RANDOM FOREST ---
=== Performance: Random Forest ===
Accuracy  : 0.7376
Precision : 0.6260
Recall    : 0.4551
F1-Score  : 0.5271
ROC-AUC   : 0.7807



In [13]:
print("--- TRAINING DECISION TREE ---")
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_processed_train, y_processed_train)
dt_pred = dt_model.predict(X_processed_val)
dt_prob = dt_model.predict_proba(X_processed_val)[:, 1]
print_evaluation_metrics("Decision Tree", y_processed_val, dt_pred, dt_prob)

--- TRAINING DECISION TREE ---
=== Performance: Decision Tree ===
Accuracy  : 0.6797
Precision : 0.5015
Recall    : 0.5095
F1-Score  : 0.5055
ROC-AUC   : 0.6350



# Train proposed ensemble models (XGBoost, LightGBM, CatBoost)

In [23]:
print("\n--- TRAINING PROPOSED ENSEMBLE (XGBoost, LightGBM, CatBoost) ---")
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
lgb = LGBMClassifier(random_state=42, verbose=-1)
cat = CatBoostClassifier(random_state=42, verbose=0)

xgb.fit(X_train, y_train)
lgb.fit(X_train, y_train)
cat.fit(X_train, y_train)

xgb_pred = xgb.predict(X_val)
lgb_pred = lgb.predict(X_val)
cat_pred = cat.predict(X_val)

prob_xgb = xgb.predict_proba(X_val)[:, 1]
prob_lgb = lgb.predict_proba(X_val)[:, 1]
prob_cat = cat.predict_proba(X_val)[:, 1]

print_evaluation_metrics("XGBoost", y_val, xgb_pred, prob_xgb)
print_evaluation_metrics("LightGBM", y_val, lgb_pred, prob_lgb)
print_evaluation_metrics("CatBoost", y_val, cat_pred, prob_cat)


--- TRAINING PROPOSED ENSEMBLE (XGBoost, LightGBM, CatBoost) ---
=== Performance: XGBoost ===
Accuracy  : 0.7352
Precision : 0.6217
Recall    : 0.4486
F1-Score  : 0.5211
ROC-AUC   : 0.7795

=== Performance: LightGBM ===
Accuracy  : 0.7388
Precision : 0.6393
Recall    : 0.4289
F1-Score  : 0.5134
ROC-AUC   : 0.7843

=== Performance: CatBoost ===
Accuracy  : 0.7404
Precision : 0.6422
Recall    : 0.4333
F1-Score  : 0.5174
ROC-AUC   : 0.7882



In [24]:
# all equal weights
w1_xgb, w1_lgb, w1_cat = 0.34, 0.33, 0.33

ensemble1_prob_val = (w1_xgb * prob_xgb) + (w1_lgb * prob_lgb) + (w1_cat * prob_cat)
ensemble1_pred_val = (ensemble1_prob_val >= 0.5).astype(int)

print_evaluation_metrics("Weighted Ensemble (Proposed)", y_val, ensemble1_pred_val, ensemble1_prob_val)

=== Performance: Weighted Ensemble (Proposed) ===
Accuracy  : 0.7408
Precision : 0.6419
Recall    : 0.4371
F1-Score  : 0.5201
ROC-AUC   : 0.7876



In [25]:
# different weight 2
w2_xgb, w2_lgb, w2_cat = 0.40, 0.15, 0.45

ensemble2_prob_val = (w2_xgb * prob_xgb) + (w2_lgb * prob_lgb) + (w2_cat * prob_cat)
ensemble2_pred_val = (ensemble2_prob_val >= 0.5).astype(int)

print_evaluation_metrics("Weighted Ensemble (Proposed)", y_val, ensemble2_pred_val, ensemble2_prob_val)

=== Performance: Weighted Ensemble (Proposed) ===
Accuracy  : 0.7405
Precision : 0.6400
Recall    : 0.4389
F1-Score  : 0.5207
ROC-AUC   : 0.7876



In [26]:
# different weight 3
w3_xgb, w3_lgb, w3_cat = 0.50, 0.15, 0.35

ensemble3_prob_val = (w3_xgb * prob_xgb) + (w3_lgb * prob_lgb) + (w3_cat * prob_cat)
ensemble3_pred_val = (ensemble3_prob_val >= 0.5).astype(int)

print_evaluation_metrics("Weighted Ensemble (Proposed)", y_val, ensemble3_pred_val, ensemble3_prob_val)

=== Performance: Weighted Ensemble (Proposed) ===
Accuracy  : 0.7402
Precision : 0.6388
Recall    : 0.4399
F1-Score  : 0.5210
ROC-AUC   : 0.7867



In [27]:
# different weight 4
w4_xgb, w4_lgb, w4_cat = 0.25, 0.20, 0.55

ensemble4_prob_val = (w4_xgb * prob_xgb) + (w4_lgb * prob_lgb) + (w4_cat * prob_cat)
ensemble4_pred_val = (ensemble4_prob_val >= 0.5).astype(int)

print_evaluation_metrics("Weighted Ensemble (Proposed)", y_val, ensemble4_pred_val, ensemble4_prob_val)

=== Performance: Weighted Ensemble (Proposed) ===
Accuracy  : 0.7410
Precision : 0.6430
Recall    : 0.4358
F1-Score  : 0.5195
ROC-AUC   : 0.7885



We will use weight 3, as it has the highest recall and f1-score meaning it has the lowest false negative rate which in our case important to minimize